# pl-jobs-lora — QLoRA fine-tune of Bielik-1.5B (S5)

Runs the training method from **ADR-0006** on a **Kaggle** GPU: zero-shot completion SFT of
Bielik-1.5B, LoRA over all linear layers, NF4 4-bit. The whole repo (dataset builder, eval harness,
probe) is driven from the local `.venv`; only *this* step needs the GPU stack
(`requirements-train.txt`). Nothing here is hardcoded — every knob comes from `configs/config.yaml`
(`train:` block).

Flow: install → auth → pull frozen dataset from HF → **Step 0** (confirm `max_seq_len`, settle the
batch split) → train + push adapter → predict base & LoRA → build the comparison report.

Before running, in the notebook's right-hand panel:

| Setting | Value | Why |
|---|---|---|
| Accelerator | **GPU T4 ×2** (or P100) | bitsandbytes needs a CUDA GPU; a 1.5B QLoRA run fits one T4 |
| Internet | **On** | `pip install`, the GitHub clone, and the HF dataset pull/adapter push |
| Add-ons → Secrets | `GH_TOKEN`, `HF_TOKEN` | the repo is private; the adapter is pushed to HF |

The free GPU session caps at ~9 h (30 h/week). Training plus three prediction variants fits, but
step 5 is the part that can run long — it is resumable, and §7 says how to carry a dropped run into
a new session.

## 1. Clone the repo + install the GPU-only training stack

In [ ]:
import os

# One visible GPU. The 1.5B fits on a single T4, and with both visible `Trainer` sees n_gpu=2 and
# wraps the model in DataParallel while bitsandbytes keeps the 4-bit weights where `device_map`
# put them — a mismatch that surfaces minutes into training, not at load.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
# Through the environment, never interpolated into the command line: a saved version keeps its
# cell output, and `https://<token>@github.com/...` in a traceback would publish the token.
os.environ["GH_TOKEN"] = secrets.get_secret("GH_TOKEN")

!git clone -q https://$GH_TOKEN@github.com/P0w3r223/pl-jobs-lora.git /kaggle/working/pl-jobs-lora
%cd /kaggle/working/pl-jobs-lora
!pip install -q -e . -r requirements-train.txt

## 2. Authenticate to HF Hub (dataset pull + adapter push) and (optionally) MLflow

In [ ]:
from huggingface_hub import login

login(token=secrets.get_secret("HF_TOKEN"))  # pull the private dataset + push the adapter

# Optional: log to the hosted MLflow (DagsHub, ADR-0004). Env var wins over config.
# os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/<user>/<repo>.mlflow'
# os.environ['MLFLOW_TRACKING_USERNAME'] = '<user>'
# os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

## 3. Pull the frozen dataset from HF into `data/processed/`
`data/processed/` is gitignored, so a fresh clone has no data — fetch the immutable S2 dataset.

In [ ]:
from pathlib import Path
from pl_jobs_lora.config import load_config
from pl_jobs_lora.dataset.hf_dataset import pull_dataset

cfg = load_config()
pull_dataset(cfg, Path('data/processed'))
print(sorted(p.name for p in Path('data/processed').glob('*')))

## Step 0 — confirm `max_seq_len`, then settle the batch split on this GPU (ADR-0006)
`train.max_seq_len` is **4096**, set on 2026-08-21 from a real-tokenizer measurement over all 568
train records (p50 3259, p95 3851, p99 4193, max 4759) — not from character counts. So this cell
re-confirms rather than discovers, and should print `OK`; run it anyway, because a different
tokenizer revision would move the distribution and truncation is silent (`encode_example` shortens
the posting, never the completion).

What is still **unconfirmed** is the batch split below it. The effective batch stays 16, but it moved
to `per_device_batch_size: 1 × grad_accum_steps: 16` when `max_seq_len` doubled, because activation
memory scales with batch × seq — and no GPU has run that. If step 4 fits with room to spare, try
`per_device 2` / `grad_accum 8` for speed; if it OOMs, per-device is already at its floor.

In [ ]:
import json, numpy as np
from transformers import AutoTokenizer
from pl_jobs_lora.train.qlora import resolve_base, to_sft_example

tok = AutoTokenizer.from_pretrained(resolve_base(cfg).hf_repo)
recs = [json.loads(l) for l in Path('data/processed/train.jsonl').read_text('utf-8').splitlines() if l.strip()]
lengths = []
for r in recs:
    sft = to_sft_example(r)
    full = sft['prompt_messages'] + [{'role': 'assistant', 'content': sft['completion']}]
    lengths.append(len(tok.apply_chat_template(full, add_generation_prompt=False)))
lengths = np.array(lengths)
print(f'tokens/example  p50={np.percentile(lengths,50):.0f}  p90={np.percentile(lengths,90):.0f}  '
      f'p99={np.percentile(lengths,99):.0f}  max={lengths.max()}')
print(f'current train.max_seq_len = {cfg.train.max_seq_len}  '
      f"({'OK' if cfg.train.max_seq_len >= np.percentile(lengths,99) else 'RAISE to cover p99'})")

## 4. Train the QLoRA adapter and push it to HF

In [ ]:
# Reload config in case you edited train.max_seq_len after Step 0.
from pl_jobs_lora.train.qlora import run_training
cfg = load_config()
adapter_dir = run_training(cfg, push=True)  # push=False to keep the adapter local first
print('adapter ->', adapter_dir)

## 5. Predict the base (adapter off) and the QLoRA adapter over the frozen test set
Same prompt + parser + greedy decoding as the API baselines (fairness). Writes
`results/eval/predictions/{bielik-1.5b__zero, bielik-1.5b__few, bielik-1.5b-lora__zero}.jsonl`.

This is the long pole — 3 × 142 greedy completions of up to `eval.max_tokens`. Each row is appended
as it is produced and a re-run skips what is on disk, so a dropped session costs one record rather
than a variant; re-run this cell to continue.

In [ ]:
from pl_jobs_lora.inference.predict_hf import run_predictions
out = run_predictions(cfg, base=True, lora=True)
for variant, preds in out.items():
    valid = sum(int(p['valid']) for p in preds)
    print(f'{variant}: {len(preds)} preds, {valid} valid JSON')

## 6. Build the comparison report (accuracy × cost × latency)
Merges these predictions with any API-baseline files present. Run the API baselines locally
(`python -m pl_jobs_lora.eval.run --baselines`) so `claude-haiku-4-5__{zero,few}.jsonl` are in
`results/eval/predictions/` for the full four-way table.

Through the CLI, not a hand-assembled `build_report` call: the CLI is where the data ceiling, the
bootstrap and the decoding cap are wired in, and a second call site here drifted out of sync with it
once already — silently, by rendering a report missing two of its sections.

In [ ]:
!python -m pl_jobs_lora.eval.run --report

## 7. Collect the artefacts
Everything under `/kaggle/working` is the notebook's **Output**, so the clone is already there —
this cell just lifts the two committable files to the top so they are easy to find after
*Save Version*. The predictions echo prose and stay gitignored, but download them anyway: they are
what lets `eval.run --report` re-score locally without paying for inference again.

A fresh Kaggle session starts with an empty `/kaggle/working`, so the resume above protects a run
*within* a session (including a restart), not across two. To continue a dropped run tomorrow, attach
this version's output as a dataset and copy `results/eval/predictions/` back before step 5.

In [ ]:
!cp results/eval/report.md results/eval/report.json /kaggle/working/
!cd results/eval && zip -qr /kaggle/working/predictions.zip predictions
print(sorted(p.name for p in Path('/kaggle/working').glob('*')))
# The adapter is already on HF (cfg.hf.adapter_repo); grab the local copy too if you want it:
# !zip -qr /kaggle/working/adapter.zip results/train/adapter